# Modelos finales · TFM Energía UCM

Entrena **una arquitectura por familia** sobre la matriz ganadora, con varias semillas, y
las deja guardadas con todo lo que hace falta para volver a usarlas.

## Por qué no vale ninguno de los notebooks anteriores

`modelo_rnn_deeplearning`, `v2_temario` y `v3_sin_series_clasicas` entrenan 15-18 modelos,
miran el resultado en pantalla y **los tiran al terminar**. Nunca se guardó ninguno. Sirven
para el relato —aquí probamos esto, salió así— no para entregar:

| | notebooks v1/v2/v3 | este |
|---|---|---|
| semillas | una | **tres**, con media ± desviación |
| guarda el `.keras` | no | **sí** |
| guarda el preprocesado | no | **sí** |
| modelos | 15-18, muchos ya descartados | **8**, uno por familia |

Y los tres entrenan los **mismos** 15 modelos: v2 añade la escalera del temario y Hyperband,
v3 quita las series clásicas. Nada de eso cambia cuál es el mejor — la escalera se queda en
13,19 frente a los 12,28 del GRU.

## Qué es una familia

Variantes de un mismo esqueleto no son familias distintas. `Seq2Seq residuo`, `+ pesos`,
`+ fine-tuning` y las seis del temario son el mismo encoder-decoder entrenado de otra
manera. Meterlas todas en un ensemble le daba a esa familia el **62 % del voto** y al GRU
—el mejor modelo individual— un 6 %. Un ensemble promedia errores *no correlacionados*:
diez copias casi idénticas no cancelan nada.

| familia | qué la distingue |
|---|---|
| `gru` | celda GRU, un tercio menos de parámetros que la LSTM |
| `conv1d_lstm` | convolución con stride antes de la recurrente |
| `seq2seq` | LSTM + RepeatVector, el de referencia |
| `simplernn` | el escalón básico, ~35 k parámetros |
| `lstm` | la gemela del GRU, para la comparación limpia |
| `denso` | MLP sobre la vista aplanada |
| `boosting` | LightGBM, un modelo por hora |
| `seq2seq_absoluto` | predice el precio, no el residuo |

El último entra **pese a perder en MAE**, porque es de los que mejor captura el spread. La
correlación entre MAE y captura es −0,285: son métricas casi independientes, y para operar
una batería manda la segunda. Elegir campeón solo por MAE elige mal para el capítulo 7.

## 0 · ¿Está usando la GPU?

Conviene comprobarlo **antes** de lanzar hora y media de entrenamiento. Que TensorFlow
*vea* la tarjeta no garantiza que la use: si falta una librería de CUDA, cae a CPU en
silencio y solo lo notas por lo lento que va.

Las tres comprobaciones de abajo, de menos a más concluyente:

1. **la ve** — `list_physical_devices` devuelve algo
2. **coloca operaciones en ella** — una multiplicación que informa dónde se ejecutó
3. **es más rápida que la CPU** — la única que descarta que esté cayendo a CPU sin avisar

In [ ]:
import tensorflow as tf, time, numpy as np

gpus = tf.config.list_physical_devices("GPU")
print(f"1. dispositivos GPU visibles: {len(gpus)}")
for g in gpus:
    det = tf.config.experimental.get_device_details(g)
    print(f"     {g.name} · {det.get('device_name', '?')} · "
          f"compute capability {det.get('compute_capability', '?')}")

if not gpus:
    print()
    print("   NO HAY GPU. Vas a entrenar por CPU: cuenta horas en lugar de minutos.")
    print("   Si estas en VS Code, revisa que el kernel sea el de WSL y no el de Windows")
    print("   -- TensorFlow >= 2.11 no usa CUDA en Windows nativo.")
else:
    # 2. ¿coloca las operaciones ahi de verdad?
    tf.debugging.set_log_device_placement(False)
    with tf.device("/GPU:0"):
        a = tf.random.normal((2000, 2000))
        _ = tf.matmul(a, a)
    print(f"\n2. una matmul de 2000x2000 se coloco en: {a.device.split('/')[-1]}")

    # 3. la prueba que de verdad descarta que este cayendo a CPU en silencio
    def cronometrar(dispositivo, n=3):
        with tf.device(dispositivo):
            x = tf.random.normal((4000, 4000))
            tf.matmul(x, x).numpy()                 # calienta, no cuenta
            t = time.perf_counter()
            for _ in range(n):
                tf.matmul(x, x).numpy()
            return (time.perf_counter() - t) / n

    t_gpu = cronometrar("/GPU:0")
    t_cpu = cronometrar("/CPU:0")
    print(f"\n3. matmul 4000x4000:  GPU {t_gpu*1000:7.1f} ms  ·  CPU {t_cpu*1000:7.1f} ms"
          f"  ->  x{t_cpu/t_gpu:.0f}")
    if t_cpu / t_gpu < 2:
        print("   SOSPECHOSO: la GPU deberia ir bastante mas rapido que eso. Puede que")
        print("   este cayendo a CPU sin avisar.")
    else:
        print("   La GPU esta trabajando.")

    memoria = tf.config.experimental.get_memory_info("GPU:0")
    print(f"\n   memoria en uso ahora: {memoria['current']/2**20:,.0f} MB "
          f"(pico {memoria['peak']/2**20:,.0f} MB)")
    print("   Con `nvidia-smi -l 2` en otra terminal lo ves en vivo mientras entrena:")
    print("   la columna GPU-Util deberia moverse entre el 30 % y el 90 %.")

In [ ]:
import sys, time, json
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
sys.path.append(str(REPO / "scripts"))

MATRIZ   = "nucleo"          # la ganadora de la comparación del notebook 05
SEMILLAS = 3
FAMILIAS = ["gru", "conv1d_lstm", "seq2seq", "simplernn", "lstm", "denso",
            "boosting", "seq2seq_absoluto"]

EJECUTAR = False             # <- ponlo a True y ejecuta esta celda

SALIDA = REPO / "data" / "gold" / f"finales_{MATRIZ}"
print(f"matriz {MATRIZ} · {len(FAMILIAS)} familias x {SEMILLAS} semillas = "
      f"{len(FAMILIAS) * SEMILLAS} entrenamientos")
print(f"salida: {SALIDA}")

## 1 · Entrenar

Cada entrenamiento escribe en `por_semilla.csv`, así que se puede parar y continuar: al
relanzar salta los que ya estén.

Con `guardar=True` cada familia deja su `.keras` **y** su `.preprocesado.json`. Lo segundo
no es opcional: un `.keras` guarda pesos, no la estandarización ni el orden de columnas.
Cargarlo sin eso devuelve números plausibles y equivocados, sin ningún aviso — el modelo
aprendió que el canal 17 es la eólica programada, y si la matriz se regenera con otro orden
sigue prediciendo tan tranquilo.

In [ ]:
GUARDAR = True

if not EJECUTAR:
    print("EJECUTAR = False. Cambia a True en la celda anterior.")
    print("Si ya lo lanzaste por terminal, salta a la sección 2: lee los mismos ficheros.")
else:
    import numpy as np, pandas as pd
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from preparar_tensores import preparar, residuo
    import entrenar_finales as EF

    print("GPU:", bool(tf.config.list_physical_devices("GPU")))
    SALIDA.mkdir(parents=True, exist_ok=True)

    T = preparar(MATRIZ)
    yr, inv_r, mu_r, sd_r = residuo(T)
    mu_y, sd_y = float(T.y[T.tr].mean()), float(T.y[T.tr].std())
    ys = ((T.y - mu_y) / sd_y).astype("float32")
    inv_abs = lambda p, m: p * sd_y + mu_y

    nv = EF.metricas(T.y[T.va], T.naive[T.va])
    nt = EF.metricas(T.y[T.te], T.naive[T.te])
    print(f"   naive: val {nv['MAE']:.2f} · test {nt['MAE']:.2f} "
          f"· captura val {nv['captura_%']:.1f}%")
    print()

    csv = SALIDA / "por_semilla.csv"
    filas = pd.read_csv(csv).to_dict("records") if csv.exists() else []
    yahay = {(r["familia"], r["semilla"]) for r in filas}
    t0 = time.time()
    for fam in FAMILIAS:
        for s in range(SEMILLAS):
            if (fam, EF.SEMILLA + s) in yahay:
                continue
            pv, pt, npar, modelo = EF.entrenar(fam, T, yr, inv_r, ys, inv_abs,
                                               EF.SEMILLA + s, keras, layers)
            mv, mt = EF.metricas(T.y[T.va], pv), EF.metricas(T.y[T.te], pt)
            filas.append({"familia": fam, "semilla": EF.SEMILLA + s, "parametros": npar,
                          "MAE_val": round(mv["MAE"], 3), "MAE_test": round(mt["MAE"], 3),
                          "captura_val_%": round(mv["captura_%"], 2),
                          "captura_test_%": round(mt["captura_%"], 2),
                          "pico_1h_test_%": round(mt["pico_1h_%"], 2),
                          "vs_naive_val_%": round(100 * (mv["MAE"] / nv["MAE"] - 1), 1)})
            pd.DataFrame(filas).to_csv(csv, index=False)
            pd.DataFrame(pv, index=pd.to_datetime(T.fechas[T.va]),
                         columns=[f"h{h:02d}" for h in range(24)]).to_csv(
                SALIDA / f"pred_val_{fam}__s{s}.csv")
            print(f"   {fam:18s} s{s}  MAE val {mv['MAE']:6.3f} · test {mt['MAE']:6.3f} "
                  f"· captura {mt['captura_%']:5.1f}%   [{(time.time()-t0)/60:.0f} min]")

            # TODAS las semillas, no solo la primera. El representante de cada familia
            # se elige despues por MAE de VALIDACION, y puede ser la s1 o la s2: guardar
            # solo la s0 exportaria un modelo distinto del que la tabla declara mejor.
            # Ocupan medio mega cada uno, asi que no hay razon para elegir antes de saber.
            if GUARDAR and modelo is not None:
                modelo.save(SALIDA / f"{fam}__s{s}.keras")
                pre = T.preprocesado()   # mismo para las 3 semillas
                pre["familia"] = fam
                pre["objetivo"] = "absoluto" if fam == "seq2seq_absoluto" else "residuo"
                pre["destipificar"] = ({"mu": mu_y, "sd": sd_y, "nota": "y = pred*sd + mu"}
                                       if fam == "seq2seq_absoluto"
                                       else {"mu": mu_r, "sd": sd_r,
                                             "nota": "y = pred*sd + mu + naive(dia D)"})
                pre["entrada"] = "plana" if fam in ("denso", "boosting") else "tensores"
                (SALIDA / f"{fam}.preprocesado.json").write_text(
                    json.dumps(pre, indent=2, ensure_ascii=False), encoding="utf-8")

    (SALIDA / "meta.json").write_text(json.dumps({
        "matriz": MATRIZ, "hash": T.meta.get("hash"), "semillas": SEMILLAS,
        "naive_val_MAE": round(nv["MAE"], 3), "naive_test_MAE": round(nt["MAE"], 3),
        "dias_train": int(T.tr.sum()), "dias_val": int(T.va.sum()),
        "dias_test": int(T.te.sum())}, indent=2), encoding="utf-8")
    print()
    print(f"Listo. {len(filas)} entrenamientos en {SALIDA}")

## 2 · Resultados por familia

Media y desviación sobre las semillas. **La desviación es la que dice si una diferencia
significa algo**: dos ejecuciones idénticas difieren hasta 0,58 en MAE, así que por debajo
de eso no se decide nada.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

csv = SALIDA / "por_semilla.csv"
if not csv.exists():
    print(f"Todavía no hay resultados en {csv}.")
    print("Ejecuta la sección 1, o lánzalo por terminal:")
    print(f"   python scripts/entrenar_finales.py --matriz {MATRIZ} "
          f"--semillas {SEMILLAS} --guardar-modelos")
    f = None
else:
    f = pd.read_csv(csv)
    meta = json.loads((SALIDA / "meta.json").read_text()) if (SALIDA / "meta.json").exists() else {}
    naive_te = meta.get("naive_test_MAE", np.nan)
    res = (f.groupby("familia").agg(
        n=("semilla", "size"), parametros=("parametros", "first"),
        MAE_val=("MAE_val", "mean"), sd_val=("MAE_val", "std"),
        MAE_test=("MAE_test", "mean"), sd_test=("MAE_test", "std"),
        captura_test=("captura_test_%", "mean"),
        pico_1h=("pico_1h_test_%", "mean")).round(3).sort_values("MAE_val"))
    res["vs_naive_%"] = (100 * (res.MAE_test / naive_te - 1)).round(1)
    display(res)
    print(f"naive de test: {naive_te} €/MWh")
    print(f"{len(f)} de {len(FAMILIAS) * SEMILLAS} entrenamientos")

HAY = f is not None and len(f) > 0

## 3 · El ensemble

Un representante por familia, **el mejor según validación**. El test no interviene en
ninguna decisión.

El anterior elegía miembros mirando el test y promediaba dieciséis modelos de los que diez
eran el mismo encoder-decoder. Este promedia familias de verdad distintas, que es lo que
hace que un ensemble funcione.

In [ ]:
if not HAY:
    print("Sin resultados todavía.")
else:
    mejor = f.loc[f.groupby("familia")["MAE_val"].idxmin()]
    print("Representante de cada familia (elegido por MAE de VALIDACIÓN):")
    display(mejor[["familia", "semilla", "MAE_val", "MAE_test", "captura_test_%"]]
            .sort_values("MAE_val").round(3))

    pv = {}
    for r in mejor.itertuples():
        p = SALIDA / f"pred_val_{r.familia}__s{r.semilla - 42}.csv"
        if p.exists():
            pv[r.familia] = pd.read_csv(p, index_col=0).values.ravel()

    if len(pv) > 1:
        print()
        print("Cuánto se parecen entre sí las predicciones de cada familia:")
        # Un ensemble gana promediando modelos que se equivocan en sitios DISTINTOS. Si dos
        # familias correlacionan a 0,99, promediarlas no cancela nada: es tener el mismo
        # modelo dos veces. Aqui se ve cuales aportan diversidad de verdad.
        c = pd.DataFrame({a: {b: np.corrcoef(x, y)[0, 1] for b, y in pv.items()}
                          for a, x in pv.items()}).round(3)
        display(c)
        import itertools
        pares = [(a, b, c.loc[a, b]) for a, b in itertools.combinations(c.index, 2)]
        pares.sort(key=lambda x: x[2])
        print(f"el par MENOS parecido : {pares[0][0]} / {pares[0][1]}  r={pares[0][2]:.3f}"
              f"   <- el que mas aporta al promedio")
        print(f"el par MAS parecido   : {pares[-1][0]} / {pares[-1][1]}  r={pares[-1][2]:.3f}")
        if pares[-1][2] > 0.98:
            print()
            print("Ese ultimo par esta por encima de 0,98: son practicamente el mismo")
            print("modelo y meter los dos en el ensemble le da doble voto a esa forma de")
            print("equivocarse.")

## 4 · MAE contra captura — no son lo mismo

El campeón por error y el campeón por dinero **no tienen por qué coincidir**, y en la
ejecución anterior no coincidían: el modelo con mejor captura del spread era el segundo peor
en MAE.

Para operar una batería no importa clavar 47,3 €/MWh: importa acertar **cuándo** cargar y
cuándo descargar. Si los dos campeones difieren, van los dos a la memoria.

In [ ]:
if not HAY:
    print("Sin resultados todavía.")
else:
    top_mae = res.index[0]
    top_cap = res.captura_test.idxmax()
    print(f"mejor MAE     : {top_mae:20s} {res.loc[top_mae,'MAE_test']:.2f} €/MWh · "
          f"captura {res.loc[top_mae,'captura_test']:.1f}%")
    print(f"mejor captura : {top_cap:20s} {res.loc[top_cap,'MAE_test']:.2f} €/MWh · "
          f"captura {res.loc[top_cap,'captura_test']:.1f}%")
    print()
    if top_mae != top_cap:
        print("NO coinciden. Exportar los dos y decirlo en la memoria: es el Hallazgo 1")
        print("del informe de fases confirmado desde el lado del deep learning.")
    else:
        print("Coinciden en esta ejecución. Aun así conviene reportar las dos métricas:")
        print("la correlación entre ambas es -0,285, así que la coincidencia es suerte.")

    fig, ax = plt.subplots(figsize=(7, 5))
    for fam, r in res.iterrows():
        ax.scatter(r.MAE_test, r.captura_test, s=110)
        ax.annotate(fam, (r.MAE_test, r.captura_test), xytext=(6, 4),
                    textcoords="offset points", fontsize=9)
    ax.set_xlabel("MAE test (€/MWh) — menos es mejor")
    ax.set_ylabel("captura del spread (%) — más es mejor")
    ax.set_title("Si fueran la misma cosa, los puntos caerían en una línea")
    ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5 · Qué queda guardado

Lo que hay en `data/gold/finales_<matriz>/` y para qué sirve cada cosa.

In [ ]:
if not SALIDA.exists():
    print("Todavía no se ha ejecutado nada.")
else:
    for p in sorted(SALIDA.iterdir()):
        kb = p.stat().st_size / 1024
        que = {".keras": "el modelo",
               ".json": "preprocesado: escaladores y orden de columnas"}.get(p.suffix, "")
        if p.name == "meta.json":
            que = "matriz, hash y listones del naive"
        elif p.name == "por_semilla.csv":
            que = "una fila por entrenamiento"
        elif p.name.startswith("pred_val_"):
            que = "predicciones de validación, 365 días x 24 h"
        print(f"  {kb:9,.0f} KB  {p.name:38s} {que}")
    print()
    print("Cada .keras necesita su .preprocesado.json para poder usarse. Al cargarlo hay")
    print("que comprobar que `hash_matriz` coincide con el de la matriz del día: si no,")
    print("fallar en voz alta -- un modelo con las columnas cambiadas de sitio sigue")
    print("prediciendo, y nadie se entera.")